# RoPE Visualization Tutorial

This notebook demonstrates how to use the RoPE visualization tools to understand Rotary Position Embeddings.

## Setup

First, install the required dependencies:

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install numpy matplotlib seaborn scikit-learn
# !pip install umap-learn  # Optional, for UMAP projections

In [ ]:
import numpy as np
import sys
from pathlib import Path

# Add rope_visualizer to path
sys.path.insert(0, str(Path.cwd().parent.parent / 'tools'))

from rope_visualizer import RopeVisualizer
from rope_visualizer.utils import apply_rope_rotation, compute_similarity_matrix

print("✅ RoPE Visualizer imported successfully!")

## 1. Generate Sample Embeddings

Let's create some random embeddings to visualize:

In [ ]:
# Configuration
hidden_dim = 128
base_theta = 10000.0
num_embeddings = 100

# Generate random embeddings
np.random.seed(42)
embeddings = np.random.randn(num_embeddings, hidden_dim).astype(np.float32)

# Normalize embeddings to unit length
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

print(f"Generated {num_embeddings} embeddings with dimension {hidden_dim}")
print(f"Embedding shape: {embeddings.shape}")

## 2. Initialize the Visualizer

Create a RopeVisualizer instance with our configuration:

In [ ]:
config = {
    'hidden_dim': hidden_dim,
    'base_theta': base_theta,
    'figsize': (12, 8)
}

viz = RopeVisualizer(config)
print("✅ Visualizer initialized")

## 3. Visualize Embedding Space (2D)

Let's project the embeddings to 2D space and see how rotation affects their distribution:

In [ ]:
# Create positions for rotation
positions = list(range(0, 100, 2))  # Every other position

# Visualize with PCA projection
viz.plot_embeddings(
    embeddings=embeddings[:50],  # Use first 50 for clarity
    positions=positions[:50],
    method='pca',
    color_by='position',
    title='RoPE Embeddings in 2D (PCA Projection)'
)

## 4. Rotation Trail

See how a single embedding moves through space as we rotate it at different positions:

In [ ]:
# Select one embedding
single_embedding = embeddings[0]

# Create positions for the trail
trail_positions = list(range(0, 200, 5))  # 0 to 195 in steps of 5

# Plot the rotation trail
viz.plot_rotation_trail(
    embedding=single_embedding,
    positions=trail_positions,
    method='pca',
    title='How One Embedding Moves with RoPE Rotation'
)

## 5. Similarity Heatmap

Visualize how similar the rotated versions are at different positions:

In [ ]:
# Select positions to compare
heatmap_positions = list(range(0, 100, 10))  # 0, 10, 20, ..., 90

# Generate similarity heatmap
viz.plot_similarity_matrix(
    embedding=single_embedding,
    positions=heatmap_positions,
    metric='cosine',
    title='Cosine Similarity Across Positions'
)

## 6. Theta Distribution

Understand the frequency spectrum of RoPE rotation pairs:

In [ ]:
viz.plot_theta_distribution(
    hidden_dim=hidden_dim,
    base_theta=base_theta,
    title='Theta Distribution Across Rotation Pairs'
)

## 7. 3D Visualization

Explore embeddings in 3D space:

In [ ]:
viz.plot_3d_embeddings(
    embeddings=embeddings[:30],  # Use 30 embeddings
    positions=list(range(0, 30)),
    method='pca',
    color_by='position',
    title='RoPE Embeddings in 3D (PCA Projection)'
)

## 8. Compare Different Projection Methods

Let's compare PCA, t-SNE, and UMAP projections side by side:

In [ ]:
import matplotlib.pyplot as plt

methods = ['pca', 'tsne']  # Add 'umap' if installed

fig, axes = plt.subplots(1, len(methods), figsize=(15, 5))

for idx, method in enumerate(methods):
    # Use viz methods but customize display
    temp_viz = RopeVisualizer(config)
    temp_viz.plot_embeddings(
        embeddings=embeddings[:40],
        positions=list(range(0, 80, 2)),
        method=method,
        title=f'{method.upper()} Projection',
        show=False
    )

plt.tight_layout()
plt.show()

## 9. Interactive Exploration

Explore different positions interactively:

In [ ]:
# Try installing ipywidgets for interactive sliders
# !pip install ipywidgets

try:
    from ipywidgets import interact, IntSlider
    
    def explore_position(position):
        """Interactive function to explore different positions"""
        # Apply rotation
        rotated = apply_rope_rotation(single_embedding, position, base_theta)
        
        # Show similarity with original
        similarity = np.dot(single_embedding, rotated)
        
        print(f"Position: {position}")
        print(f"Cosine Similarity with Original: {similarity:.4f}")
        
        # Visualize trail up to this position
        trail = list(range(0, position + 1, max(1, position // 20)))
        if trail[-1] != position:
            trail.append(position)
        
        viz.plot_rotation_trail(
            embedding=single_embedding,
            positions=trail,
            method='pca',
            title=f'Rotation Trail (0 to {position})'
        )
    
    # Create interactive slider
    interact(explore_position, position=IntSlider(min=0, max=500, step=10, value=100))
    
except ImportError:
    print("⚠️  ipywidgets not installed. Install with: pip install ipywidgets")
    print("For now, manually call explore_position(position) with different values.")

## 10. Analyze Rotation Properties

Let's verify some mathematical properties of RoPE rotation:

In [ ]:
# Test 1: Rotation preserves norm
positions_to_test = [0, 10, 50, 100, 500]
original_norm = np.linalg.norm(single_embedding)

print("Test 1: Norm Preservation")
print(f"Original norm: {original_norm:.6f}")
for pos in positions_to_test:
    rotated = apply_rope_rotation(single_embedding, pos, base_theta)
    rotated_norm = np.linalg.norm(rotated)
    print(f"  Position {pos:3d}: norm = {rotated_norm:.6f} (diff: {abs(rotated_norm - original_norm):.2e})")

# Test 2: Rotation composability
print("\nTest 2: Rotation Composability")
pos1, pos2 = 10, 20
direct = apply_rope_rotation(single_embedding, pos1 + pos2, base_theta)
composed = apply_rope_rotation(
    apply_rope_rotation(single_embedding, pos1, base_theta),
    pos2,
    base_theta
)
diff = np.linalg.norm(direct - composed)
print(f"  Rotate({pos1+pos2}) vs Rotate({pos2}) ∘ Rotate({pos1})")
print(f"  Difference: {diff:.2e} (should be ~0)")

# Test 3: Similarity decay with position
print("\nTest 3: Similarity Decay")
test_positions = [0, 10, 50, 100, 200, 500]
similarities = []
for pos in test_positions:
    rotated = apply_rope_rotation(single_embedding, pos, base_theta)
    sim = np.dot(single_embedding, rotated)
    similarities.append(sim)
    print(f"  Position {pos:3d}: similarity = {sim:.4f}")

# Plot similarity decay
plt.figure(figsize=(10, 6))
plt.plot(test_positions, similarities, 'o-', linewidth=2, markersize=8)
plt.xlabel('Position')
plt.ylabel('Cosine Similarity')
plt.title('Similarity Decay with Position')
plt.grid(True, alpha=0.3)
plt.show()

## Summary

This notebook demonstrated:
1. ✅ 2D/3D projection of rotated embeddings
2. ✅ Rotation trail visualization
3. ✅ Similarity heatmaps
4. ✅ Theta distribution analysis
5. ✅ Mathematical property verification

## Next Steps

- Try with your own embeddings from ThemisDB
- Experiment with different `base_theta` values
- Compare different embedding dimensions
- Explore learnable RoPE and LoRA-RoPE variations

For more information, see the [RoPE Visualizer README](../../tools/rope_visualizer/README.md).